In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# How to Compute the Conjunctive Normal Form

Formulas are represented as values of the type `formula`.  In order to convert a string into a formula we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [ ]:
import_notebook "Propositional-Logic-Parser.ipynb"

Furthermore, we use the module `RSet`, which is loaded by `setup.ml`.  This module implements sets in a way such that sets of sets are compared by their elements and not by their internal representation.

A formula in *conjunctive normal form* is represented in *set notation*, i.e. as a set of *clauses*, where every clause is a set of *literals*.  A literal is either a propositional variable or the negation of a propositional variable.  We define the following types:
- A `literal` is either `Pos p`, which represents the propositional variable $p$, or `Neg p`, which represents the negated variable $\neg p$.  In TypeScript, a negated variable is represented as the tuple `['¬', p]`.  In <samp>OCaml</samp>, the variant type `literal` describes precisely what a literal is.
- A `clause` is a set of literals.  It represents the disjunction of its literals.
- A `cnf` is a set of clauses.  It represents the conjunction of its clauses.

A definition of the form `type clause = literal RSet.t` defines a *type abbreviation*: `clause` is just another name for the type `literal RSet.t`.

These types are also used in the notebooks implementing the algorithm of Davis and Putnam.

In [ ]:
type literal =
  | Pos of string    (* p *)
  | Neg of string    (* ¬p *)

type clause = literal RSet.t
type cnf    = clause RSet.t

In [ ]:
let f = parse "p ↔ q"

A formula is *pure* if it contains only propositional variables, the constants `⊤` and `⊥`, and the operators `¬`, `∧`, and `∨`.  Pure formulas are represented by the type `Pure.t`.  Its constructors are the constructors of the type `formula` except for `Imp` and `Iff`.

The type is defined inside the module `Pure`.  Outside of this module, its constructors are written as `Pure.Var`, `Pure.Not`, and so on.  If we defined the constructors at the top level, they would hide the constructors of the type `formula` that have the same names.  Then `And (f, g)` would no longer be a `formula`, neither in this notebook nor in the notebooks that import it.

In [ ]:
module Pure = struct
  type t =
    | Var of string    (* propositional variable *)
    | Verum            (* ⊤ *)
    | Falsum           (* ⊥ *)
    | Not of t         (* ¬ *)
    | And of t * t     (* ∧ *)
    | Or  of t * t     (* ∨ *)
end

The function `purify` takes one argument:
- `f` is a propositional formula.

It returns a pure formula that is equivalent to `f`.  To this end, all occurrences of the operators `↔` and `→` are eliminated using the following equivalences:
$$ \begin{array}{lrcl}
   (a) & g \leftrightarrow h &\Leftrightarrow& (\neg g \vee h) \wedge (\neg h \vee g) \\
   (b) & g \rightarrow h &\Leftrightarrow& \neg g \vee h
   \end{array}
$$
The resulting formula is called a *pure* or *arrow free* formula.

**Implementation:** The subformulas of `f` are purified recursively.  The results are combined using the constructors of the type `Pure.t`.  In the case of a biconditional, the purified subformulas `gs` and `hs` are used twice.  Therefore, they are computed only once using `let ... and ... in`.

In [ ]:
let rec purify (f : formula) : Pure.t =
  match f with
  | Var p      -> Pure.Var p
  | Verum      -> Pure.Verum
  | Falsum     -> Pure.Falsum
  | Not g      -> Pure.Not (purify g)
  | And (g, h) -> Pure.And (purify g, purify h)
  | Or  (g, h) -> Pure.Or  (purify g, purify h)
  | Imp (g, h) -> Pure.Or  (Pure.Not (purify g), purify h)
  | Iff (g, h) ->
      let gs = purify g
      and hs = purify h in
      Pure.And (Pure.Or (Pure.Not gs, hs), Pure.Or (Pure.Not hs, gs))

In [ ]:
let f1 = purify f

A formula is in *negation normal form* if it is pure and the negation operator `¬` is only applied to propositional variables.  Hence, a formula in negation normal form is built from literals and the constants `⊤` and `⊥` using the operators `∧` and `∨`.  The type `NNF.t` represents these formulas:
- `NNF.Lit l` represents the literal `l`, i.e. either a propositional variable or a negated propositional variable.
- `NNF.t` has no constructor for negation.  Therefore, a value of type `NNF.t` can not contain any negation except for the negations of propositional variables that are part of literals.

The type is defined inside a module for the same reason as the type `Pure.t`.

In [ ]:
module NNF = struct
  type t =
    | Lit of literal   (* p or ¬p *)
    | Verum            (* ⊤ *)
    | Falsum           (* ⊥ *)
    | And of t * t     (* ∧ *)
    | Or  of t * t     (* ∨ *)
end

The function `nnf` takes one argument:
- `f` is a pure formula.

It returns the *negation normal form* of `f`.  The auxiliary function `neg` takes the same kind of argument `f` and returns the *negation normal form* of $\neg f$.

The function $\texttt{nnf}$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(p) = p$ for all propositional variables $p$, $\texttt{nnf}(\top) = \top$, and $\texttt{nnf}(\bot) = \bot$,</li>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) =
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\top) = \bot$ and $\texttt{neg}(\bot) = \top$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

**Implementation:** The functions `nnf` and `neg` are *mutually recursive*: each of them calls the other one.  In <samp>OCaml</samp>, a function can only call functions that have been defined before.  Mutually recursive functions are therefore defined together using `let rec ... and ...`.  `nnf` maps a propositional variable `p` to the literal `Pos p`, while `neg` maps `p` to the literal `Neg p`.

In [ ]:
let rec nnf (f : Pure.t) : NNF.t =
  match f with
  | Pure.Var p      -> NNF.Lit (Pos p)
  | Pure.Verum      -> NNF.Verum
  | Pure.Falsum     -> NNF.Falsum
  | Pure.Not g      -> neg g
  | Pure.And (g, h) -> NNF.And (nnf g, nnf h)
  | Pure.Or  (g, h) -> NNF.Or  (nnf g, nnf h)

and neg (f : Pure.t) : NNF.t =
  match f with
  | Pure.Var p      -> NNF.Lit (Neg p)
  | Pure.Verum      -> NNF.Falsum
  | Pure.Falsum     -> NNF.Verum
  | Pure.Not g      -> nnf g
  | Pure.And (g, h) -> NNF.Or  (neg g, neg h)
  | Pure.Or  (g, h) -> NNF.And (neg g, neg h)

In [ ]:
let f2 = nnf f1

The function `cnf` takes one argument:
- `f` is a formula in *negation normal form*.

It returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

**Implementation:**
- A literal $l$, i.e. a variable $p$ or a negated variable $\neg p$, is represented by the set $\bigl\{\{l\}\bigr\}$.
- $\top$ is represented by the empty set of clauses, $\bot$ by the set containing the empty clause.
- The conjunction of two formulas in CNF is the union of their clause sets.
- The disjunction of two formulas in CNF consists of all clauses $c \cup d$ where $c$ is a clause of the first formula and $d$ is a clause of the second formula.  For a fixed clause `c` of `gs`, the expression `RSet.map (RSet.union c) hs` computes the set of all clauses $c \cup d$ where $d$ is a clause of `hs`.  Here, `RSet.union c` is a *partial application*: It is the function that maps `d` to `RSet.union c d`.  `RSet.flat_map` computes the union of these sets for all clauses `c` of `gs`.

Since every value of the type `NNF.t` is in negation normal form, the pattern match covers all possible cases and `cnf` can not fail.  The type `cnf` and the function `cnf` have the same name.  This is no problem in <samp>OCaml</samp>, since types and values live in different *name spaces*.

In [ ]:
let rec cnf (f : NNF.t) : cnf =
  match f with
  | NNF.Lit l      -> RSet.singleton (RSet.singleton l)
  | NNF.Verum      -> RSet.empty
  | NNF.Falsum     -> RSet.singleton RSet.empty
  | NNF.And (g, h) -> RSet.union (cnf g) (cnf h)
  | NNF.Or  (g, h) ->
      let gs = cnf g
      and hs = cnf h in
      RSet.flat_map (fun c -> RSet.map (RSet.union c) hs) gs

In [ ]:
let f3 = cnf f2

The function `complement` takes one argument:
- `l` is a literal.

It returns the *complement* $\overline{\,l\,}$ of `l`.  If $p$ is a propositional variable, the complement is defined as follows:
 * $\overline{p} = \neg p$,
 * $\overline{\neg p} = p$.

In [ ]:
let complement (l : literal) : literal =
  match l with
  | Pos p -> Neg p
  | Neg p -> Pos p

The function `is_trivial` takes one argument:
- `c` is a clause.

It returns `true` if `c` is *trivial*, i.e. if `c` contains a literal together with its complement.  A trivial clause is a tautology.

In [ ]:
let is_trivial (c : clause) : bool =
  RSet.exists (fun l -> RSet.mem (complement l) c) c

The function `simplify` takes one argument:
- `clauses` is a set of clauses.

It returns the set of all clauses from `clauses` that are not trivial.

In [ ]:
let simplify (clauses : cnf) : cnf =
  RSet.filter (fun c -> not (is_trivial c)) clauses

We know that the following equivalence holds:
$$ C \wedge (C \vee D) \Leftrightarrow C $$
For this reason, if we have a set $S$ of clauses containing two clauses $C_1$ and $C_2$ such that
$$ C_1 \subset C_2,$$
then we can drop the clause $C_2$ from $S$ since it is *subsumed* by $C_1$.

The function `find_subsumed` takes two arguments:
- `s` is a set of clauses,
- `c1` is a clause.

It returns the set of all clauses $C_2$ from `s` that are subsumed by `c1`, i.e. all clauses `c2` that are different from `c1` and contain all literals of `c1`.  These clauses will later be removed from `s`.

In [ ]:
let find_subsumed (s : cnf) (c1 : clause) : cnf =
  RSet.filter (fun c2 -> c2 <> c1 && RSet.subset c1 c2) s

The function `subsume` takes one argument:
- `s` is a set of clauses.

It removes those clauses from `s` that are subsumed by another clause in `s`.  The partial application `find_subsumed s` maps a clause `c1` to the set of clauses subsumed by `c1`.  `RSet.flat_map` collects the clauses that are subsumed by any clause of `s`.

In [ ]:
let subsume (s : cnf) : cnf =
  RSet.diff s (RSet.flat_map (find_subsumed s) s)

The TypeScript version of this notebook defines a function `pipe` that takes a value $v$ and a list $[f_1, f_2, \cdots, f_n]$ of unary functions and returns the value $f_n(\cdots f_2(f_1(v))\cdots)$.  In <samp>OCaml</samp>, the *pipe operator* `|>` serves this purpose:
$$ v \;\texttt{|>}\; f_1 \;\texttt{|>}\; f_2 \;\texttt{|>}\; \cdots \;\texttt{|>}\; f_n = f_n(\cdots f_2(f_1(v))\cdots) $$
In TypeScript, the function `pipe` has the type `any`.  In contrast, <samp>OCaml</samp> checks that the result of every function has the type that the next function expects.

In [ ]:
3 |> (fun x -> x + 1) |> string_of_int

The function `normalize` takes one argument:
- `f` is a propositional formula.

It transforms `f` into *conjunctive normal form*.  Furthermore, both *trivial* clauses and clauses that are *subsumed* by simpler clauses are removed.

The types of the functions ensure that the transformations are applied in the right order: `purify` maps a `formula` to a `Pure.t`, `nnf` maps a `Pure.t` to an `NNF.t`, and `cnf` maps an `NNF.t` to a `cnf`.  For example, if we forgot to call `nnf`, the <samp>OCaml</samp> compiler would reject the function `normalize`.

In [ ]:
let normalize (f : formula) : cnf =
  f |> purify      (* formula -> Pure.t *)
    |> nnf         (* Pure.t  -> NNF.t  *)
    |> cnf         (* NNF.t   -> cnf    *)
    |> simplify    (* remove trivial clauses *)
    |> subsume     (* remove subsumed clauses *)

The function `prettify` takes one argument:
- `m` is a set of clauses.

It converts `m` into a string that looks like a set of sets, e.g. `"{{¬p, q}, {r}}"`.  The local function `literal_to_string` converts a literal into a string.  The keyword `function` defines a function that matches its argument against the given patterns, i.e. `function Pos p -> ... | Neg p -> ...` is short for `fun l -> match l with Pos p -> ... | Neg p -> ...`.  `RSet.to_string literal_to_string` converts a clause into a string.

In [ ]:
let prettify (m : cnf) : string =
  let literal_to_string = function
    | Pos p -> p
    | Neg p -> "¬" ^ p
  in
  RSet.to_string (RSet.to_string literal_to_string) m

The function `test` takes one argument:
- `s` is a string representing a propositional formula.

It computes the conjunctive normal form of the formula and prints it.

In [ ]:
let test (s : string) : unit =
  let f = parse s in
  Printf.printf "The CNF of %s is:\n%s\n" s (prettify (normalize f))

In [ ]:
test "(¬p → q) → (p → q) → q"

In [ ]:
test "(a → b) ↔ (¬a ∧ ¬b)"

In [ ]:
test "(p ∧ q → r) ∨ ¬r → ¬p"

In [ ]:
test "(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p"

In [ ]:
test "⊤"

In [ ]:
test "⊥"

In [ ]:
test "p → q"

In [ ]:
test "p ∧ (p → q)"

In [ ]:
test "(p ∧ q) → r"

In [ ]:
test "p ↔ q"

In [ ]:
test "(p → q) ∧ (q → r)"

In [ ]:
test "¬(p ∨ q)"

In [ ]:
test "p ∧ p"

Of course, we can also look at the clauses as values of type `cnf`:

In [ ]:
normalize (parse "(a → b) ↔ (¬a ∧ ¬b)")